In [ ]:
# Load calibration data sets DONE
# Classify gamma v. neutron DONE
# Get pulse heights, select only gamma DONE
# Make PHD DONE
# Fit gaussian to Compton edge curves
# Calculate Compton edges
# Plot PHDs
# Plot calibration curve
# Export calibration data as CSV

## Initialization

### Imports

In [ ]:
# Importing needed code

import re
import json
from collections import defaultdict
from functools import reduce
from typing import (
    Callable,
    # TypeVar,
    # Any,
    Literal
)
from datetime import datetime, timezone, timedelta
from math import sqrt, log
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
import pandas as pd
import numpy as np
from pint import Quantity
from scipy.optimize import curve_fit
from scipy.signal import (
    savgol_filter, find_peaks, peak_prominences, peak_widths
)

from data_processing.paths import (
    get_report_root, get_exp_root, get_reactor_data_root)
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    BinningDataframeColumn,
    NonReactorDataframeColumn,
    SliceFitDataframeColumn
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.loading.dataframe_loading import load_parquet_psd, load_parquet_signals
from data_processing.loading.timetag_processing import (
    calculate_timetag_hours,
    calculate_event_time
)
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
from data_processing.processing.slice_fitting import (
    get_psd_energy_histogram, scan_histogram_slices, find_failed_slices)
from data_processing.processing.calibration import Detector, recalibrate
from data_processing.processing.neutron_classification import classify
from data_processing.processing.figure_of_merit import gaussian
from data_processing.reporting.plotting import plot_scatter, plot_classification
from data_processing import helpers
from data_processing.processing.neutron_window_strategy.strategy_factory import \
    NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy import \
    AbstractNeutronStrategy
from data_processing.types import (
    NasaGenerationSettings,
    NeutronDistributionGenerationSettings,
    NeutronWindowSettings,
    WindowType,
    SliceFitStyle,
    BimodalBounds,
    BimodalParams
)
from data_processing.loading.window_loading import (
    load_side_borders, get_neutron_window_paths)
from data_processing.loading.spectrum_unfolding import load_neutron_response_matrix
from data_processing.helpers import get_midpoints_from_bins, stop
from data_processing.processing.spectrum_unfolding import NDHistogram, unfold_spectrum, _nan_divide

### Functions

In [ ]:
# fns ask questions, then generate strategy using factory

CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]


def get_nasa_loading_settings(
    calib_key: CalibrationKey
) -> str:
    left_border_type = helpers.get_input_with_default(
        """\
Which left border calculation do you want to use?
1: original left border (0.1966 MeVee)
2: newer left border (~0.1866 MeVee)
3: CAEN lower limit (0.050 MeVee) (default)
Press Enter for default
""",
        3,
        int
    )
    border_key: NasaBorderKey = (
        ExperimentDataKey.NASA_BORDERS if left_border_type == 1 
        else ExperimentDataKey.NASA_BORDERS_RECALC
    )
    file_name_prefix = f"{calib_key.value}_{border_key.value}"
    return file_name_prefix


def get_n_distro_loading_settings(
    calib_key: CalibrationKey
) -> str:
    file_name_prefix = f"{calib_key.value}_{ExperimentDataKey.N_WINDOW_BORDERS.value}"
    return file_name_prefix


def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> NasaGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = helpers.get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = helpers.get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = helpers.get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee)
2: newer (~0.1866 MeVee)
3: detector lower limit (0.050 MeVee) (default)
or press Enter for default
""",
            3,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = (
                ExperimentDataKey.NASA_BORDERS 
                if existing_left_border_version_input == 1 
                else ExperimentDataKey.NASA_BORDERS_RECALC
            )
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = get_neutron_window_paths(
                file_name_prefix=file_name_prefix)
            left_border, _ = load_side_borders(
                side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        elif existing_left_border_version_input == 3:
            lower_energy_bound = 0.05
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = helpers.get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.050)
""",
            0.050,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def get_n_distro_generation_settings(
) -> NeutronDistributionGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (3)
""",
        3,
        float
    )
    settings = NeutronDistributionGenerationSettings(
        sigma=sigma
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: NeutronStrategyFactory,
    window_type: WindowType,
    loading: bool,
    settings: NeutronWindowSettings
) -> Callable[[], AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data


In [ ]:
def moving_average(arr, n=5):
    ret = np.cumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((n-1,))
    prefix[:] = np.nan
    return np.concatenate((prefix, mov_avg))


def moving_average_centered(arr, n=5):
    if n % 2 != 1:
        raise ValueError("Centered moving average needs odd window size")
    prefix_count = (n-1)//2
    ret = np.nancumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((prefix_count,))
    suffix = np.empty((prefix_count,))
    prefix[:] = np.nan
    suffix[:] = np.nan
    return np.concatenate((prefix, mov_avg, suffix))


def simple_deriv(arr):
    hi = arr[2:]
    lo = arr[:-2]
    delta = hi - lo
    prefix = np.empty((1,))
    suffix = np.empty((1,))
    prefix[:] = np.nan
    suffix[:] = np.nan
    return np.concatenate((prefix, delta, suffix))

## Data Loading

### Loading Params

In [ ]:
isotope_datasets = {
    6: "241AmBe",
    7: "241AmBe",
    8: "241AmBe",
    22: "22Na",
    23: "22Na",
    24: "22Na",
    37: "60Co",
    38: "60Co",
    72: "137Cs",
    73: "137Cs",
    90: "152Eu",
}
isotope_datasets = {f"ID-383.{k}": v for k, v in isotope_datasets.items()}

In [ ]:
# more here?
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in isotope_datasets.keys()
}

In [ ]:
# calib_input = helpers.get_input_with_default(
#     "Do you want to use new calibration? [y/n, or press Enter for yes]",
#     "y",
#     str
# )
calib_input = "y"

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = (
    ExperimentDataKey.NEW_CALIBRATION
    if is_new_calibration
    else ExperimentDataKey.CAEN_CALIBRATION
)

In [ ]:
# detector_code = helpers.get_input_required(
#     """\
# Which detector was used?
# 1: Original detector (detector 1)
# 2: New detector (detector 2)
# """,
#     [Detector.ZERO, Detector.ONE],
#     lambda x: Detector(int(x)-1)
# )
detector_code = Detector.ZERO

In [ ]:
default_fit_input = 2  # changed to peak finder mode, approved by Fatima 2024-07-18
# fit_input = helpers.get_input_with_default(
#     """\
# Which bimodal fit type do you want to use?
# 1: Bounds based
# 2: Peak finder based (default)
# Press Enter for default
# """,
#     default_fit_input,
#     int
# )
fit_input = 2

fit_styles: dict[int, SliceFitStyle] = {
    1: "bounds",
    2: "peak_finder"
}
fit_style = fit_styles.get(fit_input, fit_styles[default_fit_input])

In [ ]:
# kind of window (Nasa, N distribution)
# load or generate
# specific settings for each condition to make namedtuple
# - generator settings (i.e. sigma, etc.)
# - file path prefix for loading
# done = False
strategy_factory = NeutronStrategyFactory()

# while not done:
#     window_input = helpers.get_input_with_default(
#         """\
# Which neutron classification window do you want to use?
# 1: NASA window (default)
# 2: Neutron distribution window
# Press Enter for default
# """,
#         1,
#         int
#     )
#     load_window_input = helpers.get_input_with_default(
#         """\
# Do you want to load the borders from the standard border file?
# [y/n, or press Enter for no]
# """,
#         "n",
#         str
#     )
#     done = True
#     will_load = load_window_input == "y"

#     try:
#         if window_input == 1:
#             if will_load:
#                 settings = get_nasa_loading_settings(calib_key=calib_key)
#                 factory_fn = make_strategy_factory_fn(
#                     strategy_factory, "nasa", True, settings
#                 )
#             else:
#                 settings = get_nasa_generation_settings(calib_key=calib_key)
#                 factory_fn = make_strategy_factory_fn(
#                     strategy_factory, "nasa", False, settings
#                 )
#                 pass
#         elif window_input == 2:
#             if will_load:
#                 settings = get_n_distro_loading_settings(calib_key=calib_key)
#                 factory_fn = make_strategy_factory_fn(
#                     strategy_factory, "n_distro", True, settings
#                 )
#             else:
#                 settings = get_n_distro_generation_settings()
#                 factory_fn = make_strategy_factory_fn(
#                     strategy_factory, "n_distro", False, settings
#                 )
#         else:
#             print("Invalid classification window type given, please try again")
#             done = False
#     except ValueError as err:
#         print("Problem found:")
#         print(err)
#         print("Please try again")
#         done = False

settings = NasaGenerationSettings(
    window_offset=0.2,
    sigma=5,
    lower_energy_bound=0.05,
    recalculate_lower_energy_bound=False
)
factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, settings
)
experiment_neutron_data = make_strategy_for_experiments(
    experiment_neutron_data, factory_fn)

### Loading

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    exp_data[ExperimentDataKey.UNCLASSIFIED] = load_parquet_psd(exp_id)
    exp_data["signals_df"] = load_parquet_signals(exp_id)

In [ ]:
# Express timetags in hours elapsed
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = calculate_timetag_hours(unclassified_df)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Recalibrate energy
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = recalibrate(unclassified_df, Detector.ZERO)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

## Processing

### Neutron Classification

In [ ]:
# Generate histogram

start_scan_idx = 0
end_scan_idx = 420
energy_width = 20e-3

for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    Z, xe, ye = get_psd_energy_histogram(
        psd_report,
        calibrated_energy_column,
        energy_width=energy_width
    )
    exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
    exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
    exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
    exp_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

In [ ]:
# TODO get fit dataframe (not needed if loading, but do anyway to keep process consistent)
stop_here = False

for exp_id, exp_data in experiment_neutron_data.items():
    Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
    xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    end_scan_idx = exp_data[ExperimentDataKey.END_SCAN_IDX]

    # # Default
    # default_bounds: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.25, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.38, 0.04, 4000)
    # )

    # bounds_a: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.04, 4000)
    # )

    # bounds_b: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.03, 4000)
    # )

    # # Ranged Example
    # bounds = [
    #     ((0, 60), bounds_a),
    # ]

    df, df_err = scan_histogram_slices(
        Z,
        xe,
        ye,
        fit_style="peak_finder",
        # default_bounds,
        # bounds=bounds,
        start_idx=start_scan_idx,
        end_idx=end_scan_idx
    )
    df, bad_slice_indexes = find_failed_slices(df, exp_id)

    if bad_slice_indexes is not None:
        exp_data[ExperimentDataKey.VALID_SLICE_FITS] = df
        exp_data[ExperimentDataKey.BAD_SLICE_INDEXES] = bad_slice_indexes
        stop_here = True
    else:
        # exp_data['fom_results'] = df
        exp_data[ExperimentDataKey.FOM_RESULTS] = df

if stop_here:
    stop()

In [ ]:
# get borders from strategy
for exp_id, exp_data in experiment_neutron_data.items():
    if ExperimentDataKey.FOM_RESULTS not in exp_data:
        print(f"No good fit data on Experiment {exp_id}")
        continue

    fom_results = exp_data[ExperimentDataKey.FOM_RESULTS]
    strategy = exp_data[ExperimentDataKey.BORDER_STRATEGY]

    strategy.set_slice_fit_dataframe(fom_results)
    borders = strategy.get_neutron_window()

    exp_data[ExperimentDataKey.BORDERS] = borders

In [ ]:
# classify neutrons
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED].copy()
    borders = exp_data[ExperimentDataKey.BORDERS]

    psd_report = classify(
        psd_report,
        calibrated_energy_column,
        borders,
        DetectorDataframeColumn.NEW_N_CLASS
    )

    exp_data[ExperimentDataKey.PSD_REPORT] = psd_report

### Pulse Height Distribution

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    n_class_col_name = DetectorDataframeColumn.NEW_N_CLASS.value
    
    gamma_only = psd_report.query(f"~{n_class_col_name}").copy()
    # neutrons_only = psd_report.query(n_class_col_name).copy()
    # exp_data[ExperimentDataKey.NEUTRONS_ONLY] = neutrons_only
    exp_data[ExperimentDataKey.GAMMA_ONLY] = gamma_only

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    gamma_only = exp_data[ExperimentDataKey.GAMMA_ONLY]
    signals_df = exp_data["signals_df"]
    g_signals_df = signals_df.loc[gamma_only.index].astype("int32")

    g_signals_np = g_signals_df.to_numpy()
    baselines = g_signals_np[:, :30].mean(axis=1).reshape(-1, 1)
    g_signals_np = -g_signals_np + baselines
    g_signals_df = pd.DataFrame(g_signals_np, index=g_signals_df.index, columns=g_signals_df.columns)
    gamma_only["peak_height"] = g_signals_df.max(axis=1)
    print(gamma_only["peak_height"].max())
    print(gamma_only["peak_height"].min())
    
    exp_data["n_signals_df"] = g_signals_df
    exp_data[ExperimentDataKey.NEUTRONS_ONLY] = gamma_only

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    gamma_only = exp_data[ExperimentDataKey.GAMMA_ONLY]
    peak_height = gamma_only["peak_height"]
    # print(neutron_energies.max())
    # print(neutron_energies.min())

    bin_start = 0
    bin_end = 15000
    bin_size = 50
    energy_bins = np.arange(bin_start, bin_end+bin_size, step=bin_size)
    
    Z, *_ = np.histogram(peak_height, bins=energy_bins)
    exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION] = {
        "gamma": {"standard": Z, "bins": energy_bins},
    }

In [ ]:
# moving average
for exp_id, exp_data in experiment_neutron_data.items():
    phd_histogram_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]
    phd_g_histogram = phd_histogram_data["gamma"]["standard"]
    # phd_g_histogram = phd_histogram_data["gamma"]["standard"]

    window_length = 37 if isotope_datasets[exp_id] == "60Co" else 17
    phd_g_moving_average = moving_average_centered(phd_g_histogram, n=window_length)
    # phd_g_moving_average = moving_average_centered(phd_g_histogram)

    phd_histogram_data["gamma"]["moving_average"] = phd_g_moving_average
    # phd_histogram_data["gamma"]["moving_average"] = phd_g_moving_average
    exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION] = phd_histogram_data

### Derivative

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    phd_histogram_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]
    phd_g_histogram = phd_histogram_data["gamma"]["moving_average"]

    phd_deriv = simple_deriv(phd_g_histogram)
    phd_histogram_data["gamma"]["derivative"] = phd_deriv

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    phd_histogram_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]
    phd_deriv = phd_histogram_data["gamma"]["derivative"]

    phd_deriv_filtered = savgol_filter(phd_deriv, 27, 3)
    phd_histogram_data["gamma"]["filter_deriv"] = phd_deriv_filtered

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    print(exp_id)
    phd_histogram_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["gamma"]
    phd_deriv_filtered = phd_histogram_data["filter_deriv"]

    isotope_name = isotope_datasets[exp_id]
    print(isotope_name)
    match isotope_name:
        case "241AmBe":
            pf_params = {
                "height": (9, 12.5),
                "prominence": 4.5,
            }
        case "22Na":
            pf_params = {
                "width": 20,
                # "prominence": 100
            }
        case "60Co":
            pf_params = {
                "prominence": 120
            }
        case "137Cs":
            pf_params = {
                "width": 20,
                "prominence": 200
            }
        case "152Eu":
            pf_params = {
                "width": 10,
                "prominence": 5
            }
        case _:
            pf_params = {}
    
    phd_deriv_filtered = -1 * phd_deriv_filtered
    peak_idx, *rest = find_peaks(phd_deriv_filtered, **pf_params)
    print(*rest)
    prominences = peak_prominences(phd_deriv_filtered, peak_idx)
    widths = peak_widths(phd_deriv_filtered, peak_idx)
    # print(prominences)
    phd_histogram_data["peak_idx"] = peak_idx
    phd_histogram_data["peak_prominences"] = prominences
    phd_histogram_data["peak_widths"] = widths

In [ ]:
test_arr = np.array([0, -1, -2, -1, 1, 2, 0, 2, 1, -12, 12, -12, 0])
expected = np.array([0, 3, 6, 8, 9, 10, 12])

signs = np.sign(test_arr)
print(signs)
zeroes, *_ = np.where(signs == 0)
print(zeroes)
left_signs = signs[:-1]
right_signs = signs[1:]
print(left_signs)
print(right_signs)
up_cross, *_ = np.where((left_signs == -1) & (right_signs == 1))
down_cross, *_ = np.where((left_signs == 1) & (right_signs == -1))
print(up_cross)
print(down_cross)
all_crosses = np.concatenate((zeroes, up_cross, down_cross))
all_crosses = np.sort(all_crosses)
print(all_crosses)

### Compton Edge

In [ ]:
# isotope_datasets = {
#     6: "241AmBe",
#     7: "241AmBe",
#     8: "241AmBe",
#     22: "22Na",
#     23: "22Na",
#     24: "22Na",
#     37: "60Co",
#     38: "60Co",
#     72: "137Cs",
#     73: "137Cs",
#     96: "152Eu",
# }
# isotope_datasets = {f"ID383.{k}": v for k, v in isotope_datasets.items()}
isotope_compton_edges = {
    "241AmBe": [4.4],
    "22Na": [0.511, 1.275],
    "60Co": [1.17, 1.33],
    "137Cs": [0.662],
    "152Eu": [0.122, 0.344, 0.779, 1.112, 1.408]
}
compton_gaussian_locs = {
    isotope_code: {
        compton_energy: [] for compton_energy in compton_energies
    }
    for isotope_code, compton_energies in isotope_compton_edges.items()
}

In [ ]:
gaussian_guesses = {
    "241AmBe": [(10000, 2000, 350)],
    "22Na": [(2500, 700, 17000), (8700, 1500, 2600)],
    "60Co": [(8000, 1500, 3700)],
    "137Cs": [(3700, 600, 7600)],
    # "152Eu": [(), (), (), (), ()]
}
gaussian_params = {}

In [ ]:
gaussian_fit_range = {  # as n sigmas after mu
    "241AmBe": [3],
    "22Na": [2, 3],
    "60Co": [3],
    "137Cs": [3],
    "152Eu": [1, 1, 1, 1, 1]
}

In [ ]:
# Fit gaussian to Compton edge curves
for exp_id, exp_data in experiment_neutron_data.items():
    phd_histogram_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["gamma"]
    Z = phd_histogram_data["standard"]
    bins = phd_histogram_data["bins"]
    isotope_code = isotope_datasets[exp_id]
    fit_guess = gaussian_guesses.get(isotope_code)

    gaussian_params[isotope_code] = fit_guess  # STUB

In [ ]:
# Calculate Compton edges

## Display

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
bg_blue = "#4c94ff"
bg_red = "#f54336"
bg_grey = "#9e9e9e"
bg_bluegrey = "#8a9fb8"

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    isotope_name = isotope_datasets[exp_id]
    print(exp_id)
    print(isotope_name)
    phd_histogram_data = exp_data[
        ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["gamma"]
    energy_bins = phd_histogram_data["bins"]
    phd_histogram = phd_histogram_data["moving_average"]
    phd_deriv = phd_histogram_data["derivative"]
    phd_deriv_filt = phd_histogram_data["filter_deriv"]
    peak_idx = phd_histogram_data["peak_idx"]
    peak_proms = phd_histogram_data["peak_prominences"]
    widths = phd_histogram_data["peak_widths"]
    
    energy_bin_mids = (energy_bins[1:] + energy_bins[:-1]) / 2
    energy_bin_widths = energy_bins[1:] - energy_bins[:-1]
    phd_histogram = np.nan_to_num(phd_histogram)
    phd_deriv = np.nan_to_num(phd_deriv)
    phd_deriv_filt = np.nan_to_num(phd_deriv_filt)
    peak_x = energy_bin_mids[peak_idx]
    peak_y = phd_deriv_filt[peak_idx]
    print(peak_x)
    print(peak_proms)
    print(widths)
    
    fig, ax = plt.subplots(figsize=(12, 8))
    ax2 = ax.twinx()

    ax.bar(energy_bin_mids, phd_histogram, width=energy_bin_widths, align="center", color=bg_blue)
    ax2.bar(energy_bin_mids, phd_deriv_filt, width=energy_bin_widths, align="center", color=bg_grey, alpha=0.5)
    ax2.plot(peak_x, peak_y, "r+", ms=20)

    match isotope_name:
        case "241AmBe":
            ax.set_ylim(0, 500)
            ax2.set_ylim(-20, 0)
        case "22Na":
            # ax2.set_ylim(-1400, 0)
            pass
        case "60Co":
            ax2.set_ylim(-150, 0)
        case "137Cs":
            ax2.set_ylim(-700, 0)
        case "152Eu":
            ax2.set_ylim(-1200, 0)
        case _:
            ax2.set_ylim(None, 0)
    # if isotope_name == "241AmBe":
    #     ax2.set_ylim(-100, 0)
    # elif isotope_name == "22Na":
    #     ax2.set_ylim(None, 0)
    # elif isotope_name == "60Co":
    #     pass
    #     # ax2.set_ylim(0, 7500)
    # elif isotope_name == "152Eu":
    #     pass

    ax.set_xlabel("Pulse height (ADC channel)", fontsize=fontsize)
    ax.set_ylabel("Counts", fontsize=fontsize)
    ax2.set_ylabel("Derivative", fontsize=fontsize)
    ax.tick_params(labelsize=fontsize)
    ax2.tick_params(labelsize=fontsize)
    plt.show()

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    isotope_name = isotope_datasets[exp_id]
    print(exp_id)
    print(isotope_name)
    neutrons_only = exp_data[ExperimentDataKey.GAMMA_ONLY]
    phd_histogram_data = exp_data[
        ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["gamma"]
    # selected_traces = exp_data["selected_traces"]
    energy_bins = phd_histogram_data["bins"]
    phd_n_histogram = phd_histogram_data["standard"]
    phd_n_moving_avg = phd_histogram_data["moving_average"]
    phd_n_moving_avg = np.nan_to_num(phd_n_moving_avg)
    fit_params_list = gaussian_params.get(isotope_name)

    energy_bin_mids = (energy_bins[1:] + energy_bins[:-1]) / 2
    energy_bin_widths = energy_bins[1:] - energy_bins[:-1]

    fig, ax1 = plt.subplots(figsize=(12, 8))

    # ax1.plot(phd_spline_y, phd_spline_x, color="black")
    # ax1.plot(phd_interp_x, phd_interp_y, color="black", linestyle="--", lw=2)
    ax1.bar(energy_bin_mids, phd_n_moving_avg, width=energy_bin_widths, align="center", color=bg_blue)
    # for i, (bin_mid, trace) in enumerate(selected_traces):
    #     trace_x = [x + i * 25 for x in range(len(trace))]
    #     ax2.plot(trace, trace_x, label=bin_mid, lw=0)
    #     ax2.fill_betweenx(trace_x, 0, trace, color=bg_blue, alpha=0.5)

    # ax2.yaxis.set_inverted(True)
    # ax1.yaxis.set_tick_params(labelleft=False, left=False)
    # ax1.xaxis.set_tick_params(labelbottom=False, bottom=False, direction="in")
    # ax2.yaxis.set_tick_params(labelleft=False, left=False)
    # ax2.xaxis.set_tick_params(bottom=False, labelbottom=False)

    # ax1.spines["top"].set_visible(False)
    # ax1.spines["right"].set_visible(False)
    # ax2.spines["bottom"].set_visible(False)
    # ax2.spines["right"].set_visible(False)

    if fit_params_list is not None:
        for fit_params in fit_params_list:
            mu, sigma, _ = fit_params
            x_min = mu - 3 * sigma
            x_max = mu + 3 * sigma
            fit_x = np.linspace(x_min, x_max, 200)
            fit_y = gaussian(fit_x, *fit_params)
            ax1.plot(fit_x, fit_y, color="black", lw=3)

    ax1.set_ylabel("Counts", fontsize=fontsize)
    ax1.set_xlabel("Pulse height (ADC channel)", fontsize=fontsize)
    if isotope_name == "241AmBe":
        ax1.set_ylim(0, 2000)
    elif isotope_name == "60Co":
        ax1.set_ylim(0, 7500)
    elif isotope_name == "152Eu":
        pass
        # ax1.set_xlim(0, 3000)
        # ax1.set_xlim(4000, 12000)
        # ax1.set_ylim(0, 4000)
        # ax1.set_xlim(4000, 7500)
        # ax1.set_ylim(2500, 4000)
        # ax1.set_yscale("log")

    # ax1.set_yscale("log")
    # ax2.set_xlabel("Pulse height (ADC channel)", fontsize=fontsize)
    # ax2.set_ylabel("Time (ns)", fontsize=fontsize)
    # ax2.xaxis.set_label_position("top")
    # ax2.xaxis.set_label_coords(0.5, 0.925)

    # limits = (0, 5000)
    # ax1.set_xlim(*limits)
    # ax2.set_xlim(*limits)
    # ax1.set_ylim(5, None)
    # ax2.set_ylim(150, 25)

    # for i, (bin_mid, trace) in enumerate(selected_traces):
    #     bin_mid_idx = np.where(energy_bin_mids == bin_mid)[0]
    #     bin_lo = float(energy_bins[bin_mid_idx][0])
    #     bin_hi = float(energy_bins[bin_mid_idx+1][0])
    #     bin_phd_x = np.linspace(bin_lo, bin_hi).reshape(-1, 1)
    #     # bin_phd_y = phd_spline(bin_phd_x).reshape(-1, 1)
    #     bin_phd_y = phd_interp(bin_phd_x).reshape(-1, 1)
    #     bin_phd_xy = np.concatenate((bin_phd_x, bin_phd_y), axis=1)
    #     trace_max_x = float(trace.idxmax()) + (i * 25)
    #     bin_trace_xy = [[bin_hi, trace_max_x], [bin_lo, trace_max_x]]
        
    #     ax1_to_display = ax1.transData.transform
    #     ax2_to_display = ax2.transData.transform
    #     display_to_figure = fig.transFigure.inverted().transform

    #     bin_phd_xy = display_to_figure(ax1_to_display(bin_phd_xy))
    #     bin_trace_xy = display_to_figure(ax2_to_display(bin_trace_xy))
    #     bin_xy = np.concatenate((bin_phd_xy, bin_trace_xy))
        
    #     poly = mpl.patches.Polygon(bin_xy, closed=True, color=bg_bluegrey, alpha=0.3)
    #     fig.add_artist(poly)
    plt.show()